In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms
from torch.utils.data import DataLoader


In [9]:
# Importing MNIST Dataset from torchvision
training_set = datasets.MNIST(root='./data', train=True, download=False, transform=transforms.ToTensor())

# Importing Validation Set from torchvision
validation_set = datasets.MNIST(root='./data', train=False, transform=transforms.ToTensor())

# Creating data loaders
training_loader = DataLoader(training_set, batch_size=32, shuffle=True)
validation_loader = DataLoader(validation_set, batch_size=32, shuffle=False)

# Class labels
classes = ('0', '1', '2', '3', '4', '5', '6', '7', '8', '9')

# Report split sizes
print(f'Training set: {len(training_set)}')
print(f'Validation set: {len(validation_set)}')

Training set: 60000
Validation set: 10000


In [20]:
class LeNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels = 1, out_channels = 6, kernel_size = 2)
        self.conv2 = nn.Conv2d(in_channels = 6, out_channels = 16, kernel_size = 2)
        self.pool = nn.AvgPool2d(2,2)
        self.fc1 = nn.Linear(576,120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10)

    def forward(self, x):
        x = self.conv1(x)
        x = F.tanh(x)
        x = self.pool(x)
        x = self.conv2(x)
        x = F.tanh(x)
        x = self.pool(x)
        x = torch.flatten(x, 1)
        x = self.fc1(x)
        x = F.tanh(x)
        x = self.fc2(x)
        x = F.tanh(x)
        x = self.fc3(x)
        return x

model = LeNet()
device ='cuda' if torch.cuda.is_available() else 'cpu'
model = model.to(device)

In [22]:
# Defining Loss Function (Since it is a multi-class classification, I chose CrossEntropyLoss)
loss_fn = nn.CrossEntropyLoss()
# Defining Optimizer
optimizer = torch.optim.SGD(model.parameters(), lr=0.001, momentum=0.9)

In [23]:
# Setup optimization loop(s)
epochs = 100

### Train time
# Loop through the epochs
for epoch in range(epochs):
    train_loss_total = 0
    test_loss_total = 0
    train_correct_guess_total = 0
    test_correct_guess_total = 0
    # Set the model to train mode (this is the default)
    model.train(True)
    for images, labels in training_loader:
        images = images.to(device)
        labels = labels.to(device)
        # 1. Do the forward pass
        y_pred = model(images)
        predicted_labels = y_pred.argmax(dim=1)
        train_correct_guess_total += (predicted_labels == labels).sum().item()
        # 2. Calculate the loss (how wrong the model is)
        loss = loss_fn(y_pred, labels)
        # 3. Zero the optimizer gradients
        optimizer.zero_grad()
        # 4. Perform backpropagation
        loss.backward()
        # 5. Step the optimizer
        optimizer.step()
        train_loss_total += loss.item() * images.size(0)
    training_accuracy = (train_correct_guess_total / len(training_set)) * 100
    train_loss_total /= len(training_set)
    ### Test time
    # Set the model to eval mode
    model.eval()
    # Turn on inference mode context manager
    with torch.inference_mode():
        for images, labels in validation_loader:
            images = images.to(device)
            labels = labels.to(device)
            # 1. Do the forward pass
            test_pred = model(images)
            predicted_labels = test_pred.argmax(dim=1)
            test_correct_guess_total += (predicted_labels == labels).sum().item()
            # 2. Calculate the loss
            test_loss = loss_fn(test_pred, labels)
            test_loss_total += test_loss.item() * images.size(0)
    test_accuracy = (test_correct_guess_total / len(validation_set)) * 100
    test_loss_total /= len(validation_set)

    # Print out what's happening
    print(f"Epoch: {epoch + 1} | Train loss: {train_loss_total:.4f} | Test loss: {test_loss_total:.4f} | Training Accuracy: {training_accuracy:.2f} % | Test Accuracy: {test_accuracy:.2f} % ")

Epoch: 1 | Train loss: 1.5191 | Test loss: 0.5265 | Training Accuracy: 51.79 % | Test Accuracy: 85.72 % 
Epoch: 2 | Train loss: 0.4153 | Test loss: 0.3382 | Training Accuracy: 88.02 % | Test Accuracy: 90.04 % 
Epoch: 3 | Train loss: 0.3126 | Test loss: 0.2727 | Training Accuracy: 90.69 % | Test Accuracy: 91.80 % 
Epoch: 4 | Train loss: 0.2621 | Test loss: 0.2315 | Training Accuracy: 92.10 % | Test Accuracy: 92.92 % 
Epoch: 5 | Train loss: 0.2250 | Test loss: 0.2030 | Training Accuracy: 93.32 % | Test Accuracy: 94.12 % 
Epoch: 6 | Train loss: 0.1960 | Test loss: 0.1858 | Training Accuracy: 94.18 % | Test Accuracy: 94.41 % 
Epoch: 7 | Train loss: 0.1739 | Test loss: 0.1594 | Training Accuracy: 94.87 % | Test Accuracy: 95.21 % 
Epoch: 8 | Train loss: 0.1558 | Test loss: 0.1451 | Training Accuracy: 95.43 % | Test Accuracy: 95.79 % 
Epoch: 9 | Train loss: 0.1416 | Test loss: 0.1343 | Training Accuracy: 95.80 % | Test Accuracy: 96.05 % 
Epoch: 10 | Train loss: 0.1293 | Test loss: 0.1217 | Tr

In [24]:
torch.save(model.state_dict(), 'lenet.pth')